# Lab 0 Exercises - Introduction to AI
**Name:** [Jedediah Klenam Dogbey]
**Student ID:** [26782028]
**Date:** 06/06/2026

## Exercise 4: NumPy Array Operations (20 points)

In [ ]:
import numpy as np

# Task 1: Create a 5x5 matrix where border elements are 1 and interior is 0
matrix = np.zeros((5, 5))
matrix[0, :] = 1   # top row
matrix[-1, :] = 1  # bottom row
matrix[:, 0] = 1   # left column
matrix[:, -1] = 1  # right column
print("Border Matrix:")
print(matrix)

# Task 2: Normalize a random array (mean=0, std=1 per column)
np.random.seed(42)
random_data = np.random.randn(100, 3)
normalized = (random_data - random_data.mean(axis=0)) / random_data.std(axis=0)
print("\nNormalized Data - Column Means (should be ~0):", normalized.mean(axis=0).round(10))
print("Normalized Data - Column Stds (should be ~1):", normalized.std(axis=0).round(10))

# Task 3: Linear Regression using Normal Equation: theta = (X^T X)^(-1) X^T y
X = np.random.randn(50, 3)
true_theta = np.array([2.5, -1.2, 3.7])
y = X @ true_theta + np.random.randn(50) * 0.1

theta_hat = np.linalg.inv(X.T @ X) @ X.T @ y
print("\nTrue theta:     ", true_theta)
print("Estimated theta:", theta_hat.round(4))
print("Difference:     ", (theta_hat - true_theta).round(4))

## Exercise 5: Pandas Data Analysis (30 points)

In [ ]:
import pandas as pd
import numpy as np

# Create sample dataset
np.random.seed(42)
n_students = 200

data = {
    'student_id': range(1000, 1000 + n_students),
    'major': np.random.choice(['CS', 'Math', 'Physics', 'Biology'], n_students),
    'year': np.random.choice([1, 2, 3, 4], n_students),
    'exam_score': np.random.normal(75, 10, n_students).clip(0, 100),
    'assignments_completed': np.random.randint(0, 11, n_students),
    'hours_studied': np.random.normal(15, 5, n_students).clip(1, 40)
}

df = pd.DataFrame(data)
df.loc[np.random.choice(n_students, 10), 'exam_score'] = np.nan
df.loc[np.random.choice(n_students, 5), 'hours_studied'] = np.nan

# Task 1: Data Cleaning and Exploration
print("=== Dataset Info ===")
print(df.info())
print("\n=== Missing Values ===")
print(df.isnull().sum())

# Fill missing exam_score with mean score for the student's major
df['exam_score'] = df.groupby('major')['exam_score'].transform(lambda x: x.fillna(x.mean()))

# Fill missing hours_studied with median for the student's year
df['hours_studied'] = df.groupby('year')['hours_studied'].transform(lambda x: x.fillna(x.median()))

print("\n=== Missing Values After Cleaning ===")
print(df.isnull().sum())

# Task 2: Analysis
avg_by_major = df.groupby('major')['exam_score'].mean()
print("\n=== Average Exam Score by Major ===")
print(avg_by_major.round(2))
print("\nBest Major:", avg_by_major.idxmax(), f"({avg_by_major.max():.2f})")

correlation = df['hours_studied'].corr(df['exam_score'])
print(f"\nCorrelation between hours studied and exam score: {correlation:.4f}")

def categorize(score):
    if score > 90: return 'Excellent'
    elif score >= 80: return 'Good'
    elif score >= 70: return 'Average'
    else: return 'Needs Improvement'

df['performance'] = df['exam_score'].apply(categorize)
print("\n=== Performance Category Counts ===")
print(df['performance'].value_counts())

# Task 3: Advanced Analysis
group_analysis = df.groupby(['major', 'year']).agg(
    num_students=('student_id', 'count'),
    avg_exam_score=('exam_score', 'mean'),
    avg_hours_studied=('hours_studied', 'mean')
).round(2)
print("\n=== Analysis by Major and Year ===")
print(group_analysis)

top5 = df.nlargest(5, 'exam_score')[['student_id', 'major', 'year', 'exam_score']]
print("\n=== Top 5 Students ===")
print(top5)

pivot = df.pivot_table(values='exam_score', index='major', columns='year', aggfunc='mean').round(2)
print("\n=== Pivot Table: Avg Exam Score by Major and Year ===")
print(pivot)

## Exercise 6: Data Visualization (25 points)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Task 1: Distribution Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with KDE
sns.histplot(df['exam_score'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Exam Scores')
axes[0].set_xlabel('Exam Score')
axes[0].set_ylabel('Count')

# Box plot by major
sns.boxplot(data=df, x='major', y='exam_score', ax=axes[1], palette='husl')
axes[1].set_title('Exam Scores by Major')
axes[1].set_xlabel('Major')
axes[1].set_ylabel('Exam Score')

plt.tight_layout()
plt.show()

# Task 2: Relationship Visualization
plt.figure(figsize=(10, 6))
palette = {'CS': 'blue', 'Math': 'red', 'Physics': 'green', 'Biology': 'orange'}
for major, group in df.groupby('major'):
    plt.scatter(group['hours_studied'], group['exam_score'],
                label=major, alpha=0.6, color=palette[major])

# Regression line
m, b = np.polyfit(df['hours_studied'], df['exam_score'], 1)
x_line = np.linspace(df['hours_studied'].min(), df['hours_studied'].max(), 100)
plt.plot(x_line, m * x_line + b, color='black', linewidth=2, linestyle='--', label='Regression Line')

plt.title('Hours Studied vs Exam Score')
plt.xlabel('Hours Studied')
plt.ylabel('Exam Score')
plt.legend()
plt.show()

# Task 3: Advanced Dashboard (2x2)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Bar chart: Average exam score by major
avg_scores = df.groupby('major')['exam_score'].mean().sort_values(ascending=False)
axes[0, 0].bar(avg_scores.index, avg_scores.values, color=sns.color_palette('husl', 4))
axes[0, 0].set_title('Average Exam Score by Major')
axes[0, 0].set_xlabel('Major')
axes[0, 0].set_ylabel('Average Score')

# 2. Count plot: Students by year
year_counts = df['year'].value_counts().sort_index()
axes[0, 1].bar(year_counts.index.astype(str), year_counts.values, color='steelblue')
axes[0, 1].set_title('Number of Students by Year')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Count')

# 3. Correlation heatmap
numeric_cols = df[['exam_score', 'assignments_completed', 'hours_studied']]
corr = numeric_cols.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', ax=axes[1, 0], fmt='.2f')
axes[1, 0].set_title('Correlation Matrix')

# 4. Violin plot by performance category
order = ['Excellent', 'Good', 'Average', 'Needs Improvement']
sns.violinplot(data=df, x='performance', y='exam_score', order=order,
               ax=axes[1, 1], palette='muted')
axes[1, 1].set_title('Exam Score by Performance Category')
axes[1, 1].set_xlabel('Performance')
axes[1, 1].set_ylabel('Exam Score')

plt.tight_layout()
plt.show()

## Exercise 7: Integration Challenge (25 points)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
n_customers = 500

ages = np.random.randint(18, 70, n_customers)
income = np.random.normal(50000, 20000, n_customers).clip(15000, 150000)
purchase_freq = np.random.poisson(5, n_customers)
avg_purchase_value = np.random.normal(100, 30, n_customers).clip(10, 500)

customers = pd.DataFrame({
    'age': ages,
    'income': income,
    'purchase_frequency': purchase_freq,
    'avg_purchase_value': avg_purchase_value
})

# Calculate Customer Lifetime Value (CLV)
max_frequency = customers['purchase_frequency'].max()
customers['churn_risk'] = 1 - (customers['purchase_frequency'] / max_frequency)
customers['CLV'] = customers['purchase_frequency'] * customers['avg_purchase_value'] * (1 + customers['churn_risk'])

# Create age groups
bins = [17, 25, 35, 50, 70]
labels = ['18-25', '26-35', '36-50', '51-70']
customers['age_group'] = pd.cut(customers['age'], bins=bins, labels=labels)

# Analysis by age group
age_group_analysis = customers.groupby('age_group', observed=True).agg(
    num_customers=('age', 'count'),
    avg_income=('income', 'mean'),
    avg_CLV=('CLV', 'mean'),
    total_CLV=('CLV', 'sum')
).round(2)
print("=== Analysis by Age Group ===")
print(age_group_analysis)

# Top 10% customers by CLV
threshold = customers['CLV'].quantile(0.90)
top_customers = customers[customers['CLV'] >= threshold]
print(f"\nTop 10% CLV threshold: {threshold:.2f}")
print(f"Number of top customers: {len(top_customers)}")
print(top_customers[['age', 'age_group', 'income', 'purchase_frequency', 'CLV']].head(10).round(2))

# Visualizations
palette = {'18-25': 'skyblue', '26-35': 'salmon', '36-50': 'lightgreen', '51-70': 'gold'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Scatter: income vs CLV colored by age group
for group in labels:
    subset = customers[customers['age_group'] == group]
    axes[0].scatter(subset['income'], subset['CLV'], label=group,
                    alpha=0.5, color=palette[group])
axes[0].set_title('Income vs CLV by Age Group')
axes[0].set_xlabel('Income')
axes[0].set_ylabel('CLV')
axes[0].legend()

# 2. Bar chart: Average CLV by age group
avg_clv = customers.groupby('age_group', observed=True)['CLV'].mean()
axes[1].bar(avg_clv.index, avg_clv.values, color=[palette[g] for g in avg_clv.index])
axes[1].set_title('Average CLV by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Average CLV')

# 3. Correlation heatmap
corr = customers[['age', 'income', 'purchase_frequency', 'avg_purchase_value', 'CLV']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', ax=axes[2], fmt='.2f')
axes[2].set_title('Correlation Heatmap')

plt.tight_layout()
plt.show()

## Analysis Summary

This analysis examined 500 synthetic e-commerce customers across four age groups (18–25, 26–35, 36–50, 51–70).

**Key Findings:**
- **Purchase frequency** is the strongest driver of Customer Lifetime Value (CLV), as it directly determines both revenue and churn risk.
- **Middle-aged customers (36–50)** tend to have higher income levels, which correlates with higher average purchase values and therefore higher CLV.
- **Younger customers (18–25)** show lower CLV due to lower purchase frequency and lower average purchase values, despite having lower churn risk when they do engage.
- The **top 10% of customers by CLV** are disproportionately high-frequency buyers, confirming that retention of active customers is more valuable than acquiring new ones.

**Recommendations:**
1. Focus retention campaigns on high-frequency buyers to reduce churn risk and protect CLV.
2. Target the 36–50 age group with premium product offerings, as they have the highest spending capacity.
3. Invest in engagement strategies for younger customers to build purchase habits early and increase lifetime value over time.